# CVAT Pose and Face Landmark Helper

Notebook này chạy trên Google Colab để giảm tải GPU máy local. Chọn `human` cho HumanPose17 hoặc `face` cho Face Landmark VF50. Mặc định chỉ tạo file JSON để kiểm tra; đặt `UPLOAD = True` sau khi đã xem preview.

In [ ]:
!pip -q install requests opencv-python-headless mediapipe ultralytics

In [ ]:
import base64, getpass, json, os, urllib.request
from pathlib import Path
import cv2, mediapipe as mp, numpy as np, requests, torch
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from ultralytics import YOLO

CVAT_URL = input('CVAT URL: ').strip().rstrip('/')
CVAT_USER = input('CVAT username: ').strip()
CVAT_PASSWORD = getpass.getpass('CVAT password: ')

In [ ]:
MODE = input('Mode [human/face]: ').strip().lower() or 'human'
UPLOAD = input('Upload? [y/n]: ').strip().lower() == 'y'  # đổi thành True chỉ sau khi đã kiểm tra preview
REPLACE_EXISTING = input('Replace existing annotations? [y/n]: ').strip().lower() == 'y'  # bắt buộc bật riêng nếu muốn thay annotation cũ
POSE_MODEL = 'yolo11l-pose.pt'  # đổi thành yolo11x-pose.pt nếu Colab đủ VRAM
POSE_IMGSZ = 1280
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
POSE_HALF = bool(torch.cuda.is_available())
assert MODE in {'human', 'face'}
print('Mode:', MODE, '| Upload:', UPLOAD)

In [ ]:
s = requests.Session()
login = s.post(CVAT_URL + '/api/auth/login', json={'username': CVAT_USER, 'password': CVAT_PASSWORD}, timeout=30)
login.raise_for_status()
s.headers.update({'Referer': CVAT_URL + '/'})
if s.cookies.get('csrftoken'):
    s.headers['X-CSRFToken'] = s.cookies['csrftoken']
jobs = s.get(CVAT_URL + '/api/jobs', params={'page': 1, 'page_size': 100}, timeout=30).json().get('results', [])
for j in jobs:
    print(f"job={j['id']} task={j.get('task_id')} frames={j.get('start_frame')}-{j.get('stop_frame')} status={j.get('status')}")
JOB_ID = int(input('Job ID: '))
job = s.get(f'{CVAT_URL}/api/jobs/{JOB_ID}', timeout=30).json()
START = int(job['start_frame']); STOP = int(job['stop_frame'])
print('Selected:', JOB_ID, START, STOP)

In [ ]:
def get_frame(frame):
    r = s.get(f'{CVAT_URL}/api/jobs/{JOB_ID}/data', params={'type':'frame','number':frame,'quality':'original'}, timeout=60)
    r.raise_for_status()
    return r.content

def point_shape(label_id, frame, xy, score=1.0, occluded=False):
    return {'label_id':label_id,'type':'points','frame':frame,'group':0,'source':'auto','score':float(score),'occluded':bool(occluded),'outside':False,'z_order':0,'rotation':0.0,'points':[float(xy[0]),float(xy[1])],'attributes':[]}

def skeleton(label_id, sub_ids, points, frame, scores=None):
    scores = scores or [1.0] * len(points)
    return {'label_id':label_id,'type':'skeleton','frame':frame,'group':0,'source':'auto','score':float(min(scores)),'occluded':False,'outside':False,'z_order':0,'rotation':0.0,'points':[],'attributes':[],'elements':[point_shape(sid, frame, p, sc, sc < 0.5) for sid,p,sc in zip(sub_ids,points,scores)]}

In [ ]:
# HumanPose17: YOLO pose trên ảnh xoay đúng chiều, sau đó đưa tọa độ về ảnh gốc.
human_model = YOLO(POSE_MODEL) if MODE == 'human' else None
def human_shapes(raw, frame):
    img = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR)
    h, w = img.shape[:2]
    rot = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    result = human_model.predict(rot, conf=0.15, imgsz=POSE_IMGSZ, device=DEVICE, half=POSE_HALF, verbose=False)[0]
    if result.keypoints is None or len(result.keypoints.xy) == 0:
        return []
    idx = int(result.boxes.xyxy[:,2:].prod(dim=1).argmax()) if len(result.boxes) else 0
    xy = result.keypoints.xy[idx].cpu().numpy()
    conf = result.keypoints.conf[idx].cpu().numpy() if result.keypoints.conf is not None else np.ones(17)
    points = [[float(y), float(h - 1 - x)] for x,y in xy]
    return [skeleton(2513, range(2514,2531), points, frame, conf.tolist())]

In [ ]:
# Face Landmark VF50: MediaPipe thử full ảnh và các crop, không phụ thuộc cv2.CascadeClassifier.
FACE_MODEL = 'face_landmarker.task'
if not Path(FACE_MODEL).exists():
    urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task', FACE_MODEL)
face_sets = [(2531,range(2532,2537),[70,63,105,66,107]),(2537,range(2538,2543),[336,296,334,293,300]),(2543,range(2544,2548),[168,6,197,195]),(2548,range(2549,2557),[33,160,158,133,153,145,144,163]),(2557,range(2558,2566),[362,385,387,263,373,374,380,382]),(2566,range(2567,2579),[61,40,39,37,0,267,269,270,291,375,321,314]),(2579,range(2580,2588),[78,191,80,13,308,324,318,95])]
def _detect_face(crop):
    rgb=cv2.cvtColor(crop,cv2.COLOR_BGR2RGB); image=mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb)
    opts=vision.FaceLandmarkerOptions(base_options=python.BaseOptions(model_asset_path=FACE_MODEL),running_mode=vision.RunningMode.IMAGE,num_faces=1,min_face_detection_confidence=.35,min_face_presence_confidence=.35)
    with vision.FaceLandmarker.create_from_options(opts) as lm: return lm.detect(image)
def face_shapes(raw, frame):
    img = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR); h,w=img.shape[:2]
    rot = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE); rh,rw=rot.shape[:2]
    # Full image first, then overlapping crops to handle the small/rotated face in VF50.
    boxes=[(0,0,rw,rh),(0,0,int(.8*rw),int(.75*rh)),(int(.2*rw),0,rw,int(.75*rh)),(0,int(.15*rh),rw,int(.9*rh)),(int(.1*rw),int(.1*rh),int(.9*rw),int(.9*rh))]
    detected=None; chosen=None
    for x0,y0,x1,y1 in boxes:
        result=_detect_face(rot[y0:y1,x0:x1])
        if result.face_landmarks:
            detected=result.face_landmarks[0]; chosen=(x0,y0,x1,y1); break
    if detected is None: return []
    x0,y0,x1,y1=chosen; f=detected
    xy=[[float(y0+p.y*(y1-y0)), float(h-1-(x0+p.x*(x1-x0)))] for p in f]
    return [skeleton(pid, sub, [xy[i] for i in inds], frame) for pid,sub,inds in face_sets]


In [ ]:
all_shapes=[]
for frame in range(START, STOP+1):
    shapes = human_shapes(get_frame(frame), frame) if MODE == 'human' else face_shapes(get_frame(frame), frame)
    print('frame', frame, 'shapes', len(shapes), 'points', sum(len(x['elements']) for x in shapes))
    all_shapes.extend(shapes)
Path(f'job_{JOB_ID}_{MODE}_annotations.json').write_text(json.dumps({'version':1,'tags':[],'shapes':all_shapes,'tracks':[]}, indent=2))
print('Saved preview JSON. No CVAT upload performed.')

In [ ]:
if UPLOAD:
    existing=s.get(f'{CVAT_URL}/api/jobs/{JOB_ID}/annotations',timeout=30).json()
    if existing.get('shapes') and not REPLACE_EXISTING:
        raise RuntimeError('Job đã có annotation. Đặt REPLACE_EXISTING=True nếu thật sự muốn thay thế.')
    r=s.put(f'{CVAT_URL}/api/jobs/{JOB_ID}/annotations',json={'version':1,'tags':[],'shapes':all_shapes,'tracks':[]},timeout=120)
    r.raise_for_status()
    print('Uploaded', len(all_shapes), 'shapes to job', JOB_ID)
else:
    print('UPLOAD=False: chỉ preview, chưa ghi CVAT')